Set project paths

In [ ]:
from pathlib import Path

def find_project_root(start_path: Path) -> Path:
    """
    Find project root folder by checking for requirements.txt and data folder.
    """
    start_path = start_path.resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path

    return start_path


PROJECT_ROOT = find_project_root(Path.cwd())

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CLEANED_DATA_PATH = PROCESSED_DIR / "cleaned_news.csv"
TRAIN_DATA_PATH = PROCESSED_DIR / "train_data.csv"
TEST_DATA_PATH = PROCESSED_DIR / "test_data.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cleaned data path:", CLEANED_DATA_PATH)
print("Train data path:", TRAIN_DATA_PATH)
print("Test data path:", TEST_DATA_PATH)

Load cleaned dataset

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

def find_project_root(start_path: Path) -> Path:
    """
    Find project root folder by checking for requirements.txt and data folder.
    """
    start_path = start_path.resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path

    return start_path


if "CLEANED_DATA_PATH" not in globals():
    PROJECT_ROOT = find_project_root(Path.cwd())
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

    CLEANED_DATA_PATH = PROCESSED_DIR / "cleaned_news.csv"
    TRAIN_DATA_PATH = PROCESSED_DIR / "train_data.csv"
    TEST_DATA_PATH = PROCESSED_DIR / "test_data.csv"

if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"cleaned_news.csv not found at: {CLEANED_DATA_PATH}\n"
        "Please run 02_preprocessing.ipynb first."
    )

data = pd.read_csv(CLEANED_DATA_PATH)

print("Cleaned dataset loaded successfully.")
print("Dataset shape:", data.shape)

print("\nColumns:")
print(data.columns.tolist())

display(data.head())

Cleaned dataset loaded successfully.
Dataset shape: (39098, 5)

Columns:
['content', 'detected_language', 'english_content', 'clean_text', 'label']


,content,detected_language,english_content,clean_text,label
0,Donald Trump Sends Out Embarrassing New Year’s...,English,Donald Trump Sends Out Embarrassing New Year’s...,donald trump sends embarrassing new year eve m...,0
1,Drunk Bragging Trump Staffer Started Russian C...,English,Drunk Bragging Trump Staffer Started Russian C...,drunk bragging trump staffer started russian c...,0
2,Sheriff David Clarke Becomes An Internet Joke ...,English,Sheriff David Clarke Becomes An Internet Joke ...,sheriff david clarke becomes internet joke thr...,0
3,Trump Is So Obsessed He Even Has Obama’s Name ...,English,Trump Is So Obsessed He Even Has Obama’s Name ...,trump obsessed even obama name coded website i...,0
4,Pope Francis Just Called Out Donald Trump Duri...,English,Pope Francis Just Called Out Donald Trump Duri...,pope francis called donald trump christmas spe...,0


Train/test split

In [4]:
from sklearn.model_selection import train_test_split

X = data["clean_text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train/test split completed.")
print("Training text shape:", X_train.shape)
print("Testing text shape:", X_test.shape)

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.0/461.0 kB 400.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 420.3 kB/s eta 0:00:0000:0100:02
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Train/test split completed.
Training text shape: (31278,)
Testing text shape: (7820,)

Training label distribution:
label
1    16957
0    14321
Name: count, dtype: int64

Testing label distribution:
label
1    4239
0    3581
Name: count, dtype: int64


Create train and test DataFrames

In [5]:
train_data = pd.DataFrame({
    "clean_text": X_train,
    "label": y_train
})

test_data = pd.DataFrame({
    "clean_text": X_test,
    "label": y_test
})

print("Train DataFrame shape:", train_data.shape)
print("Test DataFrame shape:", test_data.shape)

display(train_data.head())
display(test_data.head())

Train DataFrame shape: (31278, 2)
Test DataFrame shape: (7820, 2)


,clean_text,label
34320,catalan government appeal direct rule constitu...,1
13594,sick lie cnn luck today turn cnn day meme spre...,0
13447,cop ask mayor remove black life matter banner ...,0
6366,racist barber pull gun black man asked haircut...,0
11369,lol democrat congressman say best way fight fa...,0


,clean_text,label
25801,sept law weakens international relation saudi ...,1
2297,breaking trump top insane weekend announcing n...,0
34271,ugandan mp get work extending president rule s...,1
25990,trump belief obama born united state campaign ...,1
6788,colbert rundown harriet tubman news gonna piss...,0


Save train and test datasets

In [6]:
train_data.to_csv(TRAIN_DATA_PATH, index=False)
test_data.to_csv(TEST_DATA_PATH, index=False)

print("Train dataset saved to:", TRAIN_DATA_PATH)
print("Test dataset saved to:", TEST_DATA_PATH)

Train dataset saved to: /home/kinkini/Documents/NLP Dont delte/fake news detector/NLP_Group_05/data/processed/train_data.csv
Test dataset saved to: /home/kinkini/Documents/NLP Dont delte/fake news detector/NLP_Group_05/data/processed/test_data.csv


Verify saved files

In [7]:
saved_train_data = pd.read_csv(TRAIN_DATA_PATH)
saved_test_data = pd.read_csv(TEST_DATA_PATH)

print("Saved train dataset shape:", saved_train_data.shape)
print("Saved test dataset shape:", saved_test_data.shape)

print("\nSaved train class distribution:")
print(saved_train_data["label"].value_counts())

print("\nSaved test class distribution:")
print(saved_test_data["label"].value_counts())

display(saved_train_data.head())
display(saved_test_data.head())

Saved train dataset shape: (31278, 2)
Saved test dataset shape: (7820, 2)

Saved train class distribution:
label
1    16957
0    14321
Name: count, dtype: int64

Saved test class distribution:
label
1    4239
0    3581
Name: count, dtype: int64


,clean_text,label
0,catalan government appeal direct rule constitu...,1
1,sick lie cnn luck today turn cnn day meme spre...,0
2,cop ask mayor remove black life matter banner ...,0
3,racist barber pull gun black man asked haircut...,0
4,lol democrat congressman say best way fight fa...,0


,clean_text,label
0,sept law weakens international relation saudi ...,1
1,breaking trump top insane weekend announcing n...,0
2,ugandan mp get work extending president rule s...,1
3,trump belief obama born united state campaign ...,1
4,colbert rundown harriet tubman news gonna piss...,0
